# Part 3 · Waypoint demo (L7 with agentgateway)

**The story for the customer:** some controls only make sense at HTTP level, validating a user's JWT, allowing `GET` but not `DELETE`, splitting versions, rate-limiting. Those need a proxy in the path. In ambient that proxy is a **waypoint**, and on Solo Enterprise the waypoint is **agentgateway**, not Envoy.

This part adds the waypoint to the same petshop and layers on JWT authorisation, canary routing and identity-keyed rate limiting. It builds on Part 2's app, so if you are jumping straight here, run **Connect**, the **Part 2 context** cell, and **§2.1** (deploy petshop) first.

The division of labour stays clean: ztunnel keeps proving the caller's SPIFFE identity at L4, then hands the connection to agentgateway for L7, and that proven identity rides along as `source.identity.*` for every policy.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="720" height="250" rx="10" fill="#f8fafc"/> <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">ztunnel proves identity at L4, the waypoint enforces HTTP policy at L7</text> <defs> <marker id="wg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <!-- pipeline --> <rect x="16" y="66" width="104" height="52" rx="8" fill="#eef2ff" stroke="#6366f1"/> <text x="68" y="88" text-anchor="middle" font-size="11.5" fill="#312e81">caller</text> <text x="68" y="104" text-anchor="middle" font-size="10" fill="#4338ca">+ user JWT</text> <line x1="120" y1="92" x2="148" y2="92" stroke="#334155" stroke-width="1.6" marker-end="url(#wg)"/> <rect x="148" y="66" width="150" height="52" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="223" y="87" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">ztunnel  ·  L4</text> <text x="223" y="104" text-anchor="middle" font-size="10" fill="#475569">proves SPIFFE identity</text> <line x1="298" y1="92" x2="326" y2="92" stroke="#334155" stroke-width="1.6" marker-end="url(#wg)"/> <rect x="326" y="60" width="214" height="64" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/> <text x="433" y="83" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">waypoint (agentgateway)  ·  L7</text> <text x="433" y="100" text-anchor="middle" font-size="10" fill="#166534">JWT authN + CEL authZ</text> <text x="433" y="114" text-anchor="middle" font-size="9.5" fill="#166534">identity rides along as source.identity.*</text> <line x1="540" y1="92" x2="568" y2="92" stroke="#334155" stroke-width="1.6" marker-end="url(#wg)"/> <rect x="568" y="66" width="120" height="52" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="628" y="96" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1e293b">petstore</text> <!-- outcome pills --> <rect x="30" y="168" width="150" height="34" rx="17" fill="#fee2e2" stroke="#dc2626"/> <text x="105" y="190" text-anchor="middle" font-size="11" fill="#7f1d1d">no token → 401</text> <rect x="195" y="168" width="150" height="34" rx="17" fill="#dcfce7" stroke="#16a34a"/> <text x="270" y="190" text-anchor="middle" font-size="11" fill="#14532d">GET → 200</text> <rect x="360" y="168" width="150" height="34" rx="17" fill="#dcfce7" stroke="#16a34a"/> <text x="435" y="190" text-anchor="middle" font-size="11" fill="#14532d">DELETE (admin) → 200</text> <rect x="525" y="168" width="165" height="34" rx="17" fill="#fef3c7" stroke="#d97706"/> <text x="607" y="190" text-anchor="middle" font-size="11" fill="#7c2d12">DELETE (user) → 403</text> <text x="360" y="230" text-anchor="middle" font-size="11" fill="#64748b">Any valid token may read; only an admin may delete. One waypoint does authN, authZ, routing and rate limiting.</text> </svg></div>

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> This notebook is **self-contained**: run **Connect** first, then **Reset** for a clean slate, then the steps top to bottom.
> It needs `./demo-scripts/setup.sh` to have been run once. After a laptop sleep: `./demo-scripts/wake.sh`.

## Connect · run this first

Sets the contexts, the Solo `istioctl` build and your licence env, and confirms the platform is up. **Safe to re-run**: it only sets env, it does not touch the cluster. (Reloaded the notebook? Run just this to get your env back.)

In [ ]:
# Connect: context, image/chart versions, licences. Safe to re-run (env only).
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CTX=kind-mesh1 ISTIO_NS=istio-system TD=mesh1
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; trust domain: $TD ; licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel >/dev/null 2>&1 && echo "ambient mesh: up on ${CTX#kind-}" || echo "mesh not found on ${CTX#kind-}: run ./demo-scripts/setup.sh (or ./demo-scripts/wake.sh after a sleep)"

### Consoles / URLs

Watch traffic in the **Gloo UI** service graph. Open it once and leave it running (I open the URL in the Cursor browser):

```
./demo-scripts/consoles.sh
```

| Console | URL |
|---|---|
| Gloo UI (service graph) | http://localhost:8091 |

**Graph tips:** tick **mesh1** and the **petshop** namespace, then Graph Settings (gear) → **Idle Nodes OFF**, Traffic = **Last 1 min**, and give each change ~15-30s to show up.

From §3.2 on, an L7 denial is a real HTTP response, so denials draw as **red edges** into `petshop-waypoint`.


## Reset · clean slate

Resets the whole demo to square one (deletes the demo namespaces, reverts ztunnel), leaving the platform up so there is no rebuild. Run it **before a fresh run**, or skip it if you just reloaded and only needed the env from Connect above.

In [ ]:
# Reset the whole demo to a clean slate: deletes the demo namespaces and reverts
# ztunnel, but leaves the platform up. Run this to start fresh, NOT just for env.
bash "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/demo-scripts/reset.sh"

## 3.1 · Deploy the app and add the waypoint

**What we're doing:** demo-3 is self-contained, so we first (re)deploy the petshop app (Reset removes it), then put an agentgateway waypoint in front of it so HTTP-level policy can run.

**How:** apply the petshop workloads (idempotent), then create a `Gateway` of class `enterprise-agentgateway-waypoint` and enrol the namespace onto it with one label. `setup.sh` already installed the control plane, so this just deploys the app, creates the waypoint and points the namespace at it.

**What you'll see:** the petshop pods roll out, then the waypoint pod comes up and its own request log shows the petshop traffic now flowing through it. We reset the L4 policies first so the L7 story stands on its own (in production you would keep both, defence in depth).


<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">3.1 · One waypoint in front of the whole namespace</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="r" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="16" y="84" width="96" height="48" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="64" y="104" text-anchor="middle" font-size="11" fill="#334155">caller</text><text x="64" y="120" text-anchor="middle" font-size="8.5" fill="#475569">Bearer JWT</text><line x1="112" y1="108" x2="196" y2="108" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><rect x="198" y="80" width="196" height="64" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="296" y="100" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="296" y="116" text-anchor="middle" font-size="9" fill="#166534">class: enterprise-</text><text x="296" y="129" text-anchor="middle" font-size="9" fill="#166534">agentgateway-waypoint</text><line x1="394" y1="108" x2="466" y2="108" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><rect x="468" y="60" width="236" height="120" rx="8" fill="#ffffff" stroke="#94a3b8" stroke-width="1.4"/><text x="586" y="78" text-anchor="middle" font-size="10" font-weight="700" fill="#475569">petshop namespace</text><rect x="474" y="79" width="104" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="526" y="96" text-anchor="middle" font-size="9" fill="#1e293b">petstore</text><rect x="594" y="79" width="104" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="646" y="96" text-anchor="middle" font-size="9" fill="#1e293b">storefront</text><rect x="474" y="111" width="104" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="526" y="128" text-anchor="middle" font-size="9" fill="#1e293b">analytics</text><rect x="594" y="111" width="104" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="646" y="128" text-anchor="middle" font-size="9" fill="#1e293b">checkout-blue</text><rect x="534" y="143" width="104" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="586" y="160" text-anchor="middle" font-size="9" fill="#1e293b">checkout-green</text><text x="296" y="164" text-anchor="middle" font-size="8.5" fill="#7c2d12">one label: istio.io/use-waypoint=petshop-waypoint</text><text x="360" y="206" text-anchor="middle" font-size="10.5" fill="#64748b">One Gateway plus one namespace label: every service's L7 traffic now runs through the waypoint. ztunnel still does L4 (mTLS + identity) underneath.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# demo-3 is self-contained: (re)deploy the petshop app in case Reset removed it.
# create the namespace first (both the app and the waypoint live in it).
kubectl --context $CTX apply -f - <<'PETNS'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
PETNS

# deploy the app and wait for it in the BACKGROUND, so it rolls out while the
# waypoint comes up (apply is idempotent — a no-op if the app is already there).
kubectl --context $CTX apply -f demo-scripts/yaml/10-petshop/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s >/tmp/petshop-app.log 2>&1 &
APP_WAIT=$!

# reset the L4 policies for a clean L7 story (incl. the claims policy)
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout --ignore-not-found

# the waypoint: a Gateway of class enterprise-agentgateway-waypoint …
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petshop-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
# … then enrol the whole petshop NAMESPACE onto it (one waypoint for every service)
kubectl --context $CTX label namespace petshop istio.io/use-waypoint=petshop-waypoint --overwrite
sleep 5
kubectl --context $CTX -n petshop rollout status deploy/petshop-waypoint --timeout=150s

# the app was deploying in the background — make sure it finished too
wait $APP_WAIT; echo "petshop app: $(tail -1 /tmp/petshop-app.log)"

# proof it is agentgateway: watch its own request log as the loops flow through
kubectl --context $CTX -n petshop logs deploy/petshop-waypoint --tail=2

<details>
<summary><strong>What got created (click to expand): the waypoint the Gateway rendered</strong></summary>

You applied one `Gateway` (class `enterprise-agentgateway-waypoint`) and did not write a Deployment, yet the agentgateway controller renders a full data plane from it. On this demo the Gateway produced these objects in the `petshop` namespace:

```
$ kubectl -n petshop get deploy,svc,sa,pod -l gateway.networking.k8s.io/gateway-name=petshop-waypoint
deployment.apps/petshop-waypoint   1/1
service/petshop-waypoint           ClusterIP  10.96.121.37  15088/TCP,15008/TCP
serviceaccount/petshop-waypoint
pod/petshop-waypoint-85646bbffd-5qdm4   1/1  Running
```

The Deployment is agentgateway, not Envoy:

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: petshop-waypoint          # namespace: petshop
  labels:
    app.kubernetes.io/managed-by: agentgateway
    gateway.networking.k8s.io/gateway-class-name: enterprise-agentgateway-waypoint
    gateway.networking.k8s.io/gateway-name: petshop-waypoint
spec:
  template:
    spec:
      serviceAccountName: petshop-waypoint
      containers:
      - name: agentgateway
        image: .../agentgateway-enterprise:2026.7.0
```

The `istio.io/use-waypoint=petshop-waypoint` namespace label is what makes ztunnel send every petshop service's traffic through this proxy. The `EnterpriseAgentgatewayPolicy` and `HTTPRoute` objects in the next steps attach their L7 config to this Gateway; they do not create further data-plane objects.

See it live:
```
kubectl --context kind-mesh1 -n petshop get deploy,svc,sa,pod -l gateway.networking.k8s.io/gateway-name=petshop-waypoint
kubectl --context kind-mesh1 -n petshop get gateway petshop-waypoint -o yaml
```

</details>

## 3.2 · Authorise on the user's JWT

**What we're doing:** add user-level authorisation on top of workload identity: any signed-in user can read, only an admin can delete.

**How:** Keycloak (from `setup.sh`, realm `petshop`, users **alice**/`user` and **bob**/`admin`) issues the tokens. Two `EnterpriseAgentgatewayPolicy` objects sit on the waypoint. `jwtAuthentication` in Strict mode validates every request against Keycloak's JWKS, so no token means `401`. `authorization` then decides with CEL over the claims: any valid token may `GET`, and `DELETE` additionally needs `admin` in `realm_access.roles`.

**What you'll see:** no token → `401`, alice `GET` → `200`, alice `DELETE` → `403`, bob `DELETE` → `200`.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="300" rx="10" fill="#f8fafc"/><text x="360" y="22" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">3.2 · JWT at the waypoint: authenticate the user, then authorize the method</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="am" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker></defs><rect x="14" y="52" width="104" height="52" rx="8" fill="#e2e8f0" stroke="#64748b"/><text x="66" y="76" text-anchor="middle" font-size="11.5" font-weight="600" fill="#334155">alice · bob</text><text x="66" y="92" text-anchor="middle" font-size="8.5" fill="#475569">holds a JWT</text><text x="164" y="70" text-anchor="middle" font-size="8.5" fill="#166534">Bearer JWT + method</text><line x1="118" y1="78" x2="210" y2="78" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><rect x="212" y="44" width="250" height="88" rx="9" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="337" y="61" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">agentgateway waypoint (L7)</text><line x1="212" y1="69" x2="462" y2="69" stroke="#86efac" stroke-width="1"/><text x="224" y="85" font-size="9" font-weight="600" fill="#166534">① jwtAuthentication · Strict</text><text x="236" y="98" font-size="8" fill="#b91c1c">→ 401 if token missing / invalid</text><text x="224" y="115" font-size="9" font-weight="600" fill="#166534">② authorization · CEL over claims</text><text x="236" y="128" font-size="8" fill="#b91c1c">→ 403 if method / role not allowed</text><text x="505" y="80" text-anchor="middle" font-size="8.5" fill="#166534">allowed</text><line x1="462" y1="88" x2="548" y2="88" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><rect x="550" y="66" width="156" height="44" rx="8" fill="#dbeafe" stroke="#60a5fa"/><text x="628" y="93" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">petstore</text><rect x="212" y="146" width="250" height="30" rx="7" fill="#fef3c7" stroke="#d97706"/><text x="337" y="165" text-anchor="middle" font-size="9.5" fill="#7c2d12">Keycloak · realm petshop (JWKS)</text><line x1="337" y1="146" x2="337" y2="133" stroke="#d97706" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#am)"/><text x="470" y="160" font-size="8" fill="#92400e">JWKS → verify signature</text><rect x="14" y="190" width="692" height="102" rx="8" fill="#ffffff" stroke="#cbd5e1"/><text x="30" y="207" font-size="9" font-weight="700" fill="#475569">request</text><text x="424" y="207" font-size="9" font-weight="700" fill="#475569">result</text><text x="30" y="230" font-size="10.5" fill="#334155">no token &#183; GET /pets</text><rect x="424" y="217" width="40" height="18" rx="9" fill="#fee2e2" stroke="#dc2626"/><text x="444" y="230" text-anchor="middle" font-size="10.5" font-weight="700" fill="#b91c1c">401</text><text x="474" y="230" font-size="10" fill="#64748b">authenticate — no valid token</text><text x="30" y="249" font-size="10.5" fill="#334155">alice &#183; GET /pets</text><rect x="424" y="236" width="40" height="18" rx="9" fill="#dcfce7" stroke="#16a34a"/><text x="444" y="249" text-anchor="middle" font-size="10.5" font-weight="700" fill="#166534">200</text><text x="474" y="249" font-size="10" fill="#64748b">allowed</text><text x="30" y="268" font-size="10.5" fill="#334155">alice (user) &#183; DELETE /pets/1</text><rect x="424" y="255" width="40" height="18" rx="9" fill="#fef3c7" stroke="#d97706"/><text x="444" y="268" text-anchor="middle" font-size="10.5" font-weight="700" fill="#92400e">403</text><text x="474" y="268" font-size="10" fill="#64748b">authorize — not an admin</text><text x="30" y="287" font-size="10.5" fill="#334155">bob (admin) &#183; DELETE /pets/1</text><rect x="424" y="274" width="40" height="18" rx="9" fill="#dcfce7" stroke="#16a34a"/><text x="444" y="287" text-anchor="middle" font-size="10.5" font-weight="700" fill="#166534">200</text><text x="474" y="287" font-size="10" fill="#64748b">allowed</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
kubectl --context $CTX -n keycloak rollout status deploy/keycloak --timeout=300s
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    jwtAuthentication:
      mode: Strict
      providers:
        - issuer: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
          jwks:
            remote:
              backendRef: { name: keycloak, namespace: keycloak, port: 8080 }
              jwksPath: /realms/petshop/protocol/openid-connect/certs
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-authz, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    authorization:
      action: Allow
      policy:
        matchExpressions:
          - 'request.method == "GET"'
          - 'request.method == "DELETE" && "admin" in jwt.realm_access.roles'
EOF
sleep 10

# ── verify: both policies are attached to the waypoint ──
kubectl --context $CTX -n petshop get enterpriseagentgatewaypolicy petshop-jwt petshop-authz

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# the matrix: no token, alice (user), bob (admin).
# agentgateway answers a MISSING/invalid token with 401 (authentication),
# and a valid token that fails the CEL with 403 (authorization).
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
G=$'\e[32m'; R=$'\e[31m'; B=$'\e[1m'; C=$'\e[36m'; X=$'\e[0m'
verdict() { case "$1" in 200) printf '%s%s ✓ allowed%s' "$G$B" "$1" "$X";; 401) printf '%s%s ✗ no token%s' "$R$B" "$1" "$X";; 403) printf '%s%s ✗ not admin%s' "$R$B" "$1" "$X";; 000|"") printf '%s%s ✗ no response%s' "$R$B" "${1:-000}" "$X";; *) printf '%s' "$1";; esac; }
row() { printf "    %-30s %s\n" "$1" "$(verdict $2)"; }
printf '%s  ══ JWT at the waypoint  ·  any valid token may GET, only admin may DELETE ══%s\n' "$C$B" "$X"
row "no token      GET    /pets"    "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
row "alice         GET    /pets"    "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
row "alice (user)  DELETE /pets/1"  "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
row "bob (admin)   DELETE /pets/1"  "$(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"

**Look at the Graph now — the denials are RED.** An L7 denial is a real HTTP response, so the Graph can finally draw a block: red edges from every (token-less) client loop into `petshop-waypoint`. Contrast the L4 sections, where a denial is a reset connection and a block only ever showed as an edge going quiet. L4 deny = silence; L7 deny = red edge with a status code.

## 3.3 · Route at the waypoint: canary and header shift

**What we're doing:** show the same waypoint that checks the JWT also does traffic management, and that policy and routing compose.

**How:** attach an `HTTPRoute` whose parent is the petstore **Service** (the GAMMA pattern). Callers keep calling `petstore:8080`; the waypoint enforces a 90/10 canary to `petstore-v2` and a header shift, so `x-beta: true` goes straight to v2. The JWT policy from §3.2 still gates every request.

**What you'll see:** roughly 90/10 across v1 and v2 with a valid token, the header pinning v2, and no token still `401` on every path.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 252" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="252" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">3.3 · Route at the waypoint: 90/10 canary and an x-beta header shift</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="r" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="14" y="96" width="104" height="58" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="66" y="116" text-anchor="middle" font-size="11" fill="#334155">caller</text><text x="66" y="131" text-anchor="middle" font-size="8.5" fill="#475569">petstore:8080</text><text x="66" y="145" text-anchor="middle" font-size="8.5" fill="#475569">+ Bearer JWT</text><line x1="118" y1="124" x2="186" y2="124" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><rect x="188" y="86" width="186" height="84" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="281" y="106" text-anchor="middle" font-size="10.5" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="281" y="122" text-anchor="middle" font-size="8.5" fill="#166534">HTTPRoute petstore-split</text><text x="281" y="135" text-anchor="middle" font-size="8.5" fill="#166534">(parent: Service petstore)</text><text x="281" y="152" text-anchor="middle" font-size="8.5" fill="#b91c1c">JWT from §3.2 still gates</text><rect x="556" y="74" width="150" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="631" y="96" text-anchor="middle" font-size="11" font-weight="700" fill="#1e293b">petstore v1</text><text x="631" y="110" text-anchor="middle" font-size="8" fill="#475569">90% of default</text><rect x="556" y="150" width="150" height="46" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="631" y="172" text-anchor="middle" font-size="11" font-weight="700" fill="#7c2d12">petstore-v2</text><text x="631" y="186" text-anchor="middle" font-size="8" fill="#92400e">10% + all x-beta</text><line x1="374" y1="108" x2="554" y2="96" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><text x="470" y="92" text-anchor="middle" font-size="8.5" fill="#166534">90%</text><line x1="374" y1="140" x2="554" y2="168" stroke="#d97706" stroke-width="1.8" marker-end="url(#d)"/><text x="456" y="150" text-anchor="middle" font-size="8.5" fill="#92400e">10%</text><line x1="374" y1="152" x2="554" y2="182" stroke="#2563eb" stroke-width="1.8" stroke-dasharray="5 3" marker-end="url(#b)"/><text x="486" y="176" text-anchor="middle" font-size="8" fill="#1d4ed8">x-beta: true → v2</text><text x="360" y="226" text-anchor="middle" font-size="10.5" fill="#64748b">Same Service address; the waypoint splits it 90/10 and pins x-beta:true straight to v2. The §3.2 JWT policy still runs first, so no token is still 401.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
# petstore-v2: the same app (it reports served_by), the SAME ServiceAccount
# (routing does not change identity), its own Service.
kubectl --context $CTX apply -f demo-scripts/yaml/50-l7/30-petstore-v2.yaml
kubectl --context $CTX -n petshop rollout status deploy/petstore-v2 --timeout=120s

kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: petstore-split
  namespace: petshop
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: petstore
      port: 8080
  rules:
    - matches:
        - headers:
            - name: x-beta
              value: "true"
      backendRefs:
        - name: petstore-v2
          port: 8080
    - backendRefs:
        - name: petstore
          port: 8080
          weight: 90
        - name: petstore-v2
          port: 8080
          weight: 10
EOF
kubectl --context $CTX -n petshop get httproute petstore-split \
  -o jsonpath='{.status.parents[0].conditions[?(@.type=="Accepted")].status}{" — accepted by Service/"}{.status.parents[0].parentRef.name}' | sed "s/^True/${GRN}${BLD}True${RST}/"; echo
sleep 10

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
# the proof: distribution, header shift, and the JWT policy still in force
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "${CYN}${BLD}  ══ 20 GETs with alice's token  ·  the canary split (target 90/10) ══${RST}"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 20); do curl -s -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' \
  | sort | uniq -c | awk '{printf "    %-12s %2d / 20   (%d%%)\n", $2, $1, $1*5}'

echo
echo "${CYN}${BLD}  ══ 5 GETs with header x-beta: true  ·  pinned to v2 ══${RST}"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 5); do curl -s -m5 -H 'Authorization: Bearer $ALICE' -H 'x-beta: true' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' \
  | sort | uniq -c | awk '{printf "    %-12s %2d / 5\n", $2, $1}'

echo
NOTOK=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'x-beta: true' http://petstore:8080/pets")
echo "  no token, even on the beta route  →  ${RED}${BLD}$NOTOK${RST}   (JWT policy still gates every path)"

## 3.4 · Rate limit by workload identity

**What we're doing:** rate-limit one workload without touching another, even when both carry the same user's token. The user's token says who the **user** is; the certificate says who the **workload** is, and here they meet.

**How:** a rate limit whose CEL condition keys on `source.identity.serviceAccount`, the identity ztunnel proved at L4, enforced locally in the waypoint.

**What you'll see:** `storefront` allowed five times then `429`, while `checkout` presenting the same user JWT is untouched. The budget followed the workload's certificate, not the user's token.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 246" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="246" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">3.4 · Rate limit by workload identity, not by the user</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="r" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="14" y="74" width="132" height="50" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="80" y="94" text-anchor="middle" font-size="10.5" font-weight="700" fill="#1e293b">storefront</text><text x="80" y="110" text-anchor="middle" font-size="8" fill="#475569">carries alice's JWT</text><rect x="14" y="158" width="132" height="50" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="80" y="178" text-anchor="middle" font-size="10.5" font-weight="700" fill="#1e293b">checkout-blue</text><text x="80" y="194" text-anchor="middle" font-size="8" fill="#475569">same alice JWT</text><rect x="238" y="104" width="214" height="74" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="345" y="124" text-anchor="middle" font-size="10.5" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="345" y="140" text-anchor="middle" font-size="8.5" fill="#166534">rateLimit · conditional (CEL)</text><text x="345" y="154" text-anchor="middle" font-size="8" fill="#166534">source.identity.serviceAccount</text><text x="345" y="167" text-anchor="middle" font-size="8" fill="#166534">== "storefront"  →  5 / min</text><line x1="146" y1="104" x2="236" y2="128" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><line x1="146" y1="178" x2="236" y2="156" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><rect x="556" y="74" width="150" height="50" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="631" y="95" text-anchor="middle" font-size="10" font-weight="700" fill="#7c2d12">200 ×5, then 429</text><text x="631" y="111" text-anchor="middle" font-size="8" fill="#92400e">storefront throttled</text><rect x="556" y="158" width="150" height="50" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.4"/><text x="631" y="179" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">200 (unlimited)</text><text x="631" y="195" text-anchor="middle" font-size="8" fill="#166534">checkout untouched</text><line x1="452" y1="132" x2="554" y2="96" stroke="#d97706" stroke-width="1.8" marker-end="url(#d)"/><line x1="452" y1="150" x2="554" y2="180" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><text x="360" y="228" text-anchor="middle" font-size="10.5" fill="#64748b">Same user token on both. The limit keys on the workload's CERTIFICATE identity, so only storefront is throttled and checkout is untouched.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
# 5 requests/minute for the storefront IDENTITY, everyone else unlimited
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-ratelimit-storefront, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    rateLimit:
      conditional:
        - condition: 'source.identity.serviceAccount == "storefront"'
          policy:
            local:
              - requests: 5
                unit: Minutes
EOF
sleep 8

KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "${CYN}${BLD}  ══ 8 rapid GETs, same alice token, from two different workloads ══${RST}"
printf "    %-16s " "storefront"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done" | sed "s/200/${GRN}200${RST}/g; s/429/${YEL}429${RST}/g"; echo
printf "    %-16s " "checkout-blue"
kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done" | sed "s/200/${GRN}200${RST}/g; s/429/${YEL}429${RST}/g"; echo
echo "    limit followed the workload's certificate (storefront), not the shared user token"

`storefront: 200 ×5 then 429 429 429` — `checkout-blue: 200 ×8`, on the **same token**. The rate budget followed the workload's certificate, not the user's JWT. That is the identity thesis of this whole part, applied to L7 traffic management.

## 3.5 · Transform: propagate the verified identity as a header

**What we're doing:** §3.2 verified the JWT at the waypoint. Now put it to work: propagate that identity to the backend as a trusted `x-user` header, so the app gets the caller's identity without ever parsing a token itself.

**How:** a `transformation` on the same waypoint. It is CEL-based: read a claim from the verified token (`jwt.preferred_username`) and `set` it as a request header to the backend, and as a response header so you can see it.

**What you'll see:** the response carrying `x-authenticated-user: alice`, written by the gateway from alice's token (and `x-user: alice` added to the request the backend received).

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 226" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="226" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">3.5 · Turn the verified JWT into a trusted header</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="r" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="14" y="92" width="110" height="54" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="69" y="112" text-anchor="middle" font-size="10.5" font-weight="700" fill="#334155">caller: alice</text><text x="69" y="128" text-anchor="middle" font-size="8.5" fill="#475569">Bearer JWT</text><line x1="124" y1="110" x2="206" y2="110" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><rect x="208" y="80" width="208" height="80" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="312" y="101" text-anchor="middle" font-size="10.5" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="312" y="118" text-anchor="middle" font-size="9" fill="#166534">transformation (CEL)</text><text x="312" y="133" text-anchor="middle" font-size="8.5" fill="#166534">reads jwt.preferred_username</text><text x="312" y="148" text-anchor="middle" font-size="8.5" fill="#166534">writes it as a header</text><line x1="416" y1="102" x2="556" y2="102" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><text x="486" y="94" text-anchor="middle" font-size="8" fill="#166534">request + x-user: alice</text><rect x="558" y="80" width="148" height="54" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="632" y="100" text-anchor="middle" font-size="11" font-weight="700" fill="#1e293b">petstore</text><text x="632" y="115" text-anchor="middle" font-size="8" fill="#475569">sees x-user: alice</text><text x="632" y="128" text-anchor="middle" font-size="8" fill="#475569">(never parses a token)</text><line x1="556" y1="140" x2="416" y2="140" stroke="#2563eb" stroke-width="1.8" stroke-dasharray="5 3" marker-end="url(#b)"/><text x="486" y="152" text-anchor="middle" font-size="8" fill="#1d4ed8">response + x-authenticated-user: alice</text><text x="360" y="204" text-anchor="middle" font-size="10.5" fill="#64748b">The gateway reads the verified claim and writes it as a header, so the backend gets the caller's identity without ever touching a token.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# §3.4 left a 5/min rate limit on storefront; clear it so this transform test (also from
# storefront) isn't rate-limited.
kubectl --context $CTX -n petshop delete enterpriseagentgatewaypolicy petshop-ratelimit-storefront --ignore-not-found
# CEL transformation on the waypoint: a verified JWT claim -> header (request + response)
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-transform, namespace: petshop }
spec:
  targetRefs: [{ group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }]
  traffic:
    transformation:
      request:
        set:
          - name: x-user
            value: "jwt.preferred_username"       # verified identity, forwarded to the backend
      response:
        set:
          - name: x-authenticated-user
            value: "jwt.preferred_username"       # proof, visible to the caller
EOF
sleep 8
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')
echo "  == alice's response now carries the gateway-injected identity =="
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -i -m6 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets" 2>/dev/null \
  | grep -iE 'HTTP/|x-authenticated-user'
echo "    (x-user: alice was also added to the REQUEST the backend received, derived from the JWT)"

## 3.6 · Delegate authz another way: OPA and ext-auth (reference)

§3.2 authorized with **CEL** written straight into the `EnterpriseAgentgatewayPolicy` (any valid token may `GET`, only an admin may `DELETE`). CEL in the CRD is the zero-infrastructure default, and most teams never leave it. When you need **external data**, **policy-as-code you can unit-test**, or a **custom auth service**, the waypoint delegates the decision instead. Same outcome, decided outside the gateway. Full write-up: [CEL vs OPA vs ext-authz vs ext-proc](https://www.masterthemesh.com/solo/blog/cel-vs-opa-extauth-extproc/).

The two options below are shown as reference; they are not run in this notebook.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 258" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="258" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">3.6 · Where the authz decision runs: CEL in the gateway, or delegated</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="r" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="20" y="46" width="150" height="44" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="95" y="65" text-anchor="middle" font-size="9.5" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="95" y="79" text-anchor="middle" font-size="8" fill="#166534">§3.2 default</text><line x1="170" y1="68" x2="298" y2="68" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><text x="235" y="60" text-anchor="middle" font-size="7.5" fill="#475569">in-CRD</text><rect x="300" y="46" width="232" height="44" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.6"/><text x="416" y="64" text-anchor="middle" font-size="9.5" font-weight="700" fill="#0f172a">CEL in the EnterpriseAgentgatewayPolicy</text><text x="416" y="78" text-anchor="middle" font-size="8.5" fill="#0f172a">zero infra · decided in the gateway</text><rect x="590" y="57" width="112" height="22" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/><text x="646" y="72" text-anchor="middle" font-size="8.5" font-weight="700" fill="#14532d">allow / deny</text><rect x="20" y="116" width="150" height="44" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="95" y="135" text-anchor="middle" font-size="9.5" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="95" y="149" text-anchor="middle" font-size="8" fill="#166534">Option A · extAuth</text><line x1="170" y1="138" x2="298" y2="138" stroke="#d97706" stroke-width="1.8" marker-end="url(#d)"/><text x="235" y="130" text-anchor="middle" font-size="7.5" fill="#475569">extAuth gRPC</text><rect x="300" y="116" width="232" height="44" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="416" y="134" text-anchor="middle" font-size="9.5" font-weight="700" fill="#0f172a">OPA (Rego) · opa-envoy gRPC</text><text x="416" y="148" text-anchor="middle" font-size="8.5" fill="#0f172a">external data · unit-testable</text><rect x="590" y="127" width="112" height="22" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/><text x="646" y="142" text-anchor="middle" font-size="8.5" font-weight="700" fill="#14532d">allow / deny</text><rect x="20" y="186" width="150" height="44" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="95" y="205" text-anchor="middle" font-size="9.5" font-weight="700" fill="#14532d">petshop-waypoint</text><text x="95" y="219" text-anchor="middle" font-size="8" fill="#166534">Option B · entExtAuth</text><line x1="170" y1="208" x2="298" y2="208" stroke="#2563eb" stroke-width="1.8" marker-end="url(#b)"/><text x="235" y="200" text-anchor="middle" font-size="7.5" fill="#475569">entExtAuth</text><rect x="300" y="186" width="232" height="44" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.6"/><text x="416" y="204" text-anchor="middle" font-size="9.5" font-weight="700" fill="#0f172a">Solo ext-auth · AuthConfig</text><text x="416" y="218" text-anchor="middle" font-size="8.5" fill="#0f172a">chained plugins: jwt && opa</text><rect x="590" y="197" width="112" height="22" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/><text x="646" y="212" text-anchor="middle" font-size="8.5" font-weight="700" fill="#14532d">allow / deny</text><text x="360" y="244" text-anchor="middle" font-size="10.5" fill="#64748b">Same allow/deny. CEL in the CRD is the default; delegate to OPA or Solo ext-auth when you need external data, policy-as-code, or a custom auth service.</text></svg></div>

<details>
<summary><strong>Option A: OPA (Rego) as a direct ext-authz service</strong></summary>

Run OPA with the opa-envoy-plugin (it speaks the Envoy ext-authz gRPC protocol) and a Rego policy, then point the waypoint's `extAuth` straight at it. OPA re-reads the JWT and makes the same decision in Rego, external to the gateway and unit-testable.

```yaml
# Rego (opa-envoy input model: input.attributes.request.http.*)
package petshop.authz
import future.keywords.if
import future.keywords.in
default allow := false
allow if input.attributes.request.http.method == "GET"
allow if {
  input.attributes.request.http.method == "DELETE"
  auth := input.attributes.request.http.headers.authorization
  [_, payload, _] := io.jwt.decode(substring(auth, 7, -1))   # strip "Bearer "
  "admin" in payload.realm_access.roles
}
```

```yaml
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-opa, namespace: petshop }
spec:
  targetRefs: [{ group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }]
  traffic:
    extAuth:
      backendRef: { name: opa-authz, namespace: petshop, port: 9191 }   # OPA opa-envoy, gRPC
      grpc: {}
```

Same allow/deny matrix as §3.2, now in Rego. Note: the `opa-envoy` image is published for **amd64 only**, so run this on an amd64 cluster (it will not pull on an Apple-Silicon kind).

</details>



<details>
<summary><strong>Option B: Solo ext-auth service + AuthConfig (chained plugins)</strong></summary>

Solo ships an ext-auth service (already running here as `ext-auth-service-enterprise-agentgateway`) with a plugin chain: OIDC, OAuth2 introspection, JWT, OPA, API-key, LDAP, HMAC, custom gRPC. You write an `AuthConfig` and point the waypoint at the service with `entExtAuth`. `booleanExpr` chains plugins, so one config can require a valid token **and** an OPA decision, and even fold a claim into a header with `claimsToHeaders` (the §3.5 transformation, done inside ext-auth).

```yaml
apiVersion: extauth.solo.io/v1
kind: AuthConfig
metadata: { name: petshop-jwt-and-opa, namespace: agentgateway-system }
spec:
  booleanExpr: "jwt && opa"
  configs:
    - name: jwt
      oauth2:
        accessTokenValidation:
          jwt:
            remoteJwks:
              url: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/certs
            issuer: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
          claimsToHeaders:
            - { claim: preferred_username, header: x-user }
    - name: opa
      opaAuth:
        modules:
          - { name: petshop-rego, namespace: agentgateway-system }   # Rego in a ConfigMap
        query: "data.authz.allow == true"
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-entextauth, namespace: petshop }
spec:
  targetRefs: [{ group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }]
  traffic:
    entExtAuth:
      authConfigRef: { name: petshop-jwt-and-opa, namespace: agentgateway-system }
      backendRef:
        name: ext-auth-service-enterprise-agentgateway
        namespace: agentgateway-system
        port: 8083
```

The ext-auth service does JWT validation and the OPA decision in one chained `AuthConfig`. The OPA plugin's Rego uses the Solo ext-auth input model (`input.http_request.*`, `input.state[...]`), not the opa-envoy model in Option A. Deep dive: the [Solo external auth](/solo/solo-ext-auth/) KB.

</details>

## Tear down and move on

Finished with this part? Run the cell below to start cleaning up, then switch to the next lab and talk it through while the cluster resets in the background. It deletes the demo namespaces and reverts ztunnel, but leaves the platform (clusters, ambient mesh, peering, agentgateway, Gloo UI, Keycloak) up, so the next lab needs no rebuild. It is the same reset the next lab's setup cell runs, so it is safe to run anytime.

In [ ]:
# Tear down this part while you switch to the next lab: deletes the demo namespaces
# and reverts ztunnel, but leaves the platform up. Safe to run anytime.
bash "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/demo-scripts/reset.sh"